# Khái niệm và Trực quan hóa các Đặc trưng Hình ảnh trong Thị giác Máy tính

Notebook này giải thích chi tiết các khái niệm:
1. **BGR vs HSV** (Không gian màu sắc)
2. **LBP** (Local Binary Pattern - Kết cấu bề mặt)
3. **Hu Moments** (Mô-men hình dáng toàn cục)
4. **HOG** (Histogram of Oriented Gradients - Mô tả hướng cạnh biên cục bộ)

Chúng ta sẽ tạo một hình ảnh giả lập (synthetic image) mô tả một chiếc xe để thực hiện trực quan hóa từng bước.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
from skimage import feature
from skimage.feature import hog

# 1. Tạo ảnh xe giả lập để chạy thử nghiệm
def create_synthetic_car():
    # Tạo canvas trắng 128x128 pixel
    img = np.ones((128, 128, 3), dtype=np.uint8) * 255
    
    # Vẽ cabin xe (Màu xanh dương - BGR: 255, 0, 0)
    cv2.rectangle(img, (40, 30), (88, 60), (255, 0, 0), -1)
    
    # Vẽ thân xe (Màu đỏ - BGR: 0, 0, 255)
    cv2.rectangle(img, (20, 60), (108, 90), (0, 0, 255), -1)
    
    # Vẽ 2 bánh xe (Màu xám đậm - BGR: 50, 50, 50)
    cv2.circle(img, (40, 95), 15, (50, 50, 50), -1)
    cv2.circle(img, (88, 95), 15, (50, 50, 50), -1)
    
    # Vẽ mâm xe (Màu sáng - BGR: 200, 200, 200) để tạo kết cấu bánh xe
    cv2.circle(img, (40, 95), 5, (200, 200, 200), -1)
    cv2.circle(img, (88, 95), 5, (200, 200, 200), -1)
    
    # Thêm một ít kết cấu đốm (nhiễu màu xanh lá cây) trên thân xe
    np.random.seed(42)
    for _ in range(30):
        x = np.random.randint(22, 106)
        y = np.random.randint(62, 88)
        cv2.circle(img, (x, y), 1, (0, 255, 0), -1)
        
    return img

car_bgr = create_synthetic_car()
car_rgb = cv2.cvtColor(car_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(4, 4))
plt.imshow(car_rgb)
plt.title("Ảnh xe giả lập (RGB)")
plt.axis('off')
plt.show()

ModuleNotFoundError: No module named 'cv2'

---
## I. BGR vs HSV (Không gian màu sắc)

### 1. Khái niệm cơ bản
* **BGR (Blue - Green - Red):** Là không gian màu cộng tính mặc định của OpenCV. Nó biểu diễn một màu sắc bằng cách pha trộn 3 kênh màu cơ bản là Xanh dương (B), Xanh lá (G) và Đỏ (R) với giá trị từ 0 đến 255.
* **HSV (Hue - Saturation - Value):** Là không gian màu mô phỏng cách con người cảm nhận màu sắc:
  * **Hue (H - Tông màu):** Giá trị từ 0° đến 360° (OpenCV chuẩn hóa thành 0-180). Đại diện cho loại màu sắc thực tế (Đỏ, Cam, Vàng, Lục, Lam, Tím...).
  * **Saturation (S - Độ bão hòa):** Từ 0 đến 255. Đại diện cho độ thuần khiết/đậm nhạt của màu (càng nhỏ màu càng nhạt/xám, càng lớn màu càng tươi/rực).
  * **Value (V - Giá trị/Độ sáng):** Từ 0 đến 255. Đại diện cho mức độ sáng tối của màu (0 là đen kịt, 255 là sáng nhất).

### 2. Sự ảnh hưởng lên dữ liệu học máy
* Trong **BGR**, khi cường độ ánh sáng thay đổi (ví dụ: trời nắng gắt hoặc bóng râm), **cả 3 giá trị B, G, R đều biến động mạnh** cùng lúc. Điều này khiến mô hình học máy rất khó nhận diện màu sắc của xe một cách nhất quán.
* Trong **HSV**, thông tin màu sắc chính nằm ở kênh **H (Hue)** và **S (Saturation)**. Kênh **V (Value)** chứa thông tin về ánh sáng. Do đó, để phân loại màu xe độc lập với thời tiết/ánh sáng, ta có thể tập trung phân tích kênh **H** và **S**, bỏ qua hoặc giảm bins của kênh **V**.

In [ ]:
# Chuyển sang HSV
car_hsv = cv2.cvtColor(car_bgr, cv2.COLOR_BGR2HSV)

# Tách các kênh BGR và HSV
b, g, r = cv2.split(car_bgr)
h, s, v = cv2.split(car_hsv)

# Trực quan hóa
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

# Dòng 1: Ảnh gốc và các kênh BGR
axes[0, 0].imshow(car_rgb)
axes[0, 0].set_title("Ảnh gốc (RGB)")
axes[0, 0].axis('off')

axes[0, 1].imshow(b, cmap='Blues')
axes[0, 1].set_title("Kênh Blue (B)")
axes[0, 1].axis('off')

axes[0, 2].imshow(g, cmap='Greens')
axes[0, 2].set_title("Kênh Green (G)")
axes[0, 2].axis('off')

axes[0, 3].imshow(r, cmap='Reds')
axes[0, 3].set_title("Kênh Red (R)")
axes[0, 3].axis('off')

# Dòng 2: Ảnh gốc và các kênh HSV
axes[1, 0].imshow(car_rgb)
axes[1, 0].set_title("Ảnh gốc (RGB)")
axes[1, 0].axis('off')

axes[1, 1].imshow(h, cmap='hsv')
axes[1, 1].set_title("Kênh Hue (H - Tông màu)")
axes[1, 1].axis('off')

axes[1, 2].imshow(s, cmap='gray')
axes[1, 2].set_title("Kênh Saturation (S - Độ đậm)")
axes[1, 2].axis('off')

axes[1, 3].imshow(v, cmap='gray')
axes[1, 3].set_title("Kênh Value (V - Độ sáng)")
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

---
## II. LBP (Local Binary Pattern - Đặc trưng kết cấu)

### 1. Khái niệm và Cách hoạt động từng bước
LBP được dùng để mô tả kết cấu bề mặt cục bộ (nhám, mịn, sọc, tròn...):
1. Với mỗi pixel trong ảnh xám, ta lấy nó làm **tâm** ($P_c$) và xét các pixel lân cận trong bán kính $R$.
2. So sánh giá trị pixel lân cận ($P_n$) với tâm:
   * Nếu $P_n \ge P_c$: gán giá trị **1**.
   * Nếu $P_n < P_c$: gán giá trị **0**.
3. Đi vòng quanh tâm theo chiều kim đồng hồ để thu được chuỗi 8 bit nhị phân (ví dụ: `11000101`).
4. Chuyển chuỗi nhị phân đó thành một số thập phân (từ 0 đến 255). Số này chính là giá trị LBP mới của pixel tâm đó.
* **Uniform LBP:** Giảm số chiều bằng cách chỉ đếm các mẫu có tối đa 2 lần chuyển đổi giữa `0` và `1` (ví dụ: `00000000` chuyển đổi 0 lần, `00011100` chuyển đổi 2 lần - đạt chuẩn; còn `01010101` chuyển đổi quá nhiều lần - quy về một bin chung).

### 2. Sự ảnh hưởng lên dữ liệu học máy
* LBP giúp mô hình nhận diện được các chi tiết bề mặt như vân bánh xe, lưới tản nhiệt, họa tiết mâm xe mà không phụ thuộc vào màu sắc của xe.
* Giá trị LBP có tính bất biến với sự thay đổi độ sáng tuyến tính (vì so sánh hiệu số lớn hơn/nhỏ hơn giữa pixel lân cận và tâm).

In [ ]:
# Chuyển ảnh sang ảnh xám
car_gray = cv2.cvtColor(car_bgr, cv2.COLOR_BGR2GRAY)

# Trích xuất LBP đồng nhất (Uniform LBP)
num_points = 24
radius = 3
lbp_map = feature.local_binary_pattern(car_gray, num_points, radius, method="uniform")

# Tính toán histogram của LBP
n_bins = num_points + 2
hist_lbp, _ = np.histogram(lbp_map.ravel(), bins=n_bins, range=(0, n_bins), density=True)

# Vẽ hình ảnh LBP và biểu đồ
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(car_gray, cmap='gray')
axes[0].set_title("Ảnh xám gốc")
axes[0].axis('off')

axes[1].imshow(lbp_map, cmap='gray')
axes[1].set_title(f"Bản đồ LBP (P={num_points}, R={radius})")
axes[1].axis('off')

axes[2].bar(range(n_bins), hist_lbp, color='darkblue', edgecolor='black')
axes[2].set_title("LBP Histogram (Vector 26 chiều)")
axes[2].set_xlabel("Bin thứ")
axes[2].set_ylabel("Mật độ")

plt.tight_layout()
plt.show()

---
## III. Hu Moments (Mô-men hình khối)

### 1. Khái niệm và Cách hoạt động từng bước
Hu Moments là tập hợp gồm **7 số thực** được tính toán từ các mô-men không gian trung tâm của một hình ảnh nhị phân (chỉ gồm 0 và 255):
1. **Phân ngưỡng nhị phân (Thresholding):** Chuyển ảnh xám thành ảnh nhị phân để tách đối tượng cần nhận dạng (màu trắng) ra khỏi nền (màu đen).
2. **Tính các mô-men chuẩn hóa không đổi (Normalized Central Moments):** Tính toán tích phân phân bố khối lượng pixel để loại bỏ các ảnh hưởng của phép dịch chuyển (Translation) và phép thu phóng tỉ lệ (Scaling).
3. **Tính 7 mô-men Hu:** Kết hợp các mô-men trung tâm qua các công thức toán học phi tuyến để tạo ra 7 giá trị bất biến với phép quay (Rotation).

### 2. Sự ảnh hưởng của phân ngưỡng nhị phân lên dữ liệu
* **Ngưỡng tĩnh 128 (Static Threshold):** Nếu ảnh chụp phương tiện bị tối quá hoặc sáng quá, ngưỡng 128 sẽ biến toàn bộ xe thành đen xì hoặc trắng xóa, làm mất hoàn toàn hình dạng thật của xe.
* **Ngưỡng Otsu (Otsu's Threshold):** Tự động phân tích biểu đồ phân bố độ xám để tìm ra ngưỡng tối ưu phân cách giữa nền và xe độc lập với độ sáng tổng thể.

In [ ]:
# Tạo một ảnh tối hơn để kiểm tra tính ổn định của phân ngưỡng
dark_car_gray = np.clip(car_gray.astype(int) - 100, 0, 255).astype(np.uint8)

# 1. Sử dụng ngưỡng tĩnh 128 trên ảnh tối
_, thresh_fixed = cv2.threshold(dark_car_gray, 128, 255, cv2.THRESH_BINARY)

# 2. Sử dụng ngưỡng tự động Otsu trên ảnh tối
otsu_thresh_val, thresh_otsu = cv2.threshold(dark_car_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Vẽ hình để so sánh
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].imshow(dark_car_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Ảnh xám bị tối")
axes[0].axis('off')

axes[1].imshow(thresh_fixed, cmap='gray')
axes[1].set_title("Ngưỡng tĩnh 128 (Bị hỏng hình dáng)")
axes[1].axis('off')

axes[2].imshow(thresh_otsu, cmap='gray')
axes[2].set_title(f"Ngưỡng tự động Otsu (Ngưỡng tìm được: {int(otsu_thresh_val)})")
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Tính toán Hu Moments
moments = cv2.moments(thresh_otsu)
hu_raw = cv2.HuMoments(moments).flatten()
# Log-transform
hu_log = []
for val in hu_raw:
    if val != 0:
        hu_log.append(-1 * math.copysign(1.0, val) * math.log10(abs(val)))
    else:
        hu_log.append(0.0)

print("7 mô-men Hu (sau khi biến đổi Log để thu nhỏ trị số):")
for i, h_val in enumerate(hu_log):
    print(f"  Hu[{i+1}]: {h_val:.4f}")

---
## IV. HOG (Histogram of Oriented Gradients - Đặc trưng hướng cạnh biên)

### 1. Khái niệm và Cách hoạt động từng bước
HOG cực kỳ mạnh mẽ trong việc nhận diện cấu trúc hình học của vật thể (đặc biệt là xe cộ và con người):
1. **Tính đạo hàm/độ dốc (Gradients):** Tính toán sự thay đổi cường độ sáng theo trục X và Y cho từng pixel. Điều này giúp phát hiện ra các cạnh biên (edges).
2. **Phân chia thành các Cell:** Chia ảnh thành các ô nhỏ (ví dụ: $16 \times 16$ pixel mỗi ô).
3. **Tính toán Histogram của hướng độ dốc:** Trong mỗi ô, đếm sự xuất hiện của các hướng cạnh biên (từ 0° đến 180°, chia thành 9 phần/bins).
4. **Chuẩn hóa theo Block:** Gom các ô cạnh nhau lại thành khối lớn hơn (ví dụ: khối gồm $2 \times 2$ ô) để chuẩn hóa độ sáng cục bộ, giúp chống nhiễu do bóng đổ.

### 2. Sự ảnh hưởng lên dữ liệu học máy
* HOG giữ lại rất tốt hình dáng cạnh bao quanh của xe (dáng nằm ngang, bánh xe tròn, kính chắn gió xiên), giúp mô hình Random Forest học được "bản vẽ thiết kế" hình dạng xe để phân biệt cực kỳ chuẩn xác.

In [ ]:
# Tính đặc trưng HOG và lấy ảnh trực quan hóa
hog_features, hog_image = hog(
    car_gray, 
    orientations=9, 
    pixels_per_cell=(8, 8), 
    cells_per_block=(2, 2), 
    visualize=True,
    block_norm='L2-Hys'
)

# Trực quan hóa HOG
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(car_rgb)
axes[0].set_title("Ảnh xe gốc")
axes[0].axis('off')

# Cường điệu hóa cường độ ảnh HOG để nhìn rõ các hướng vạch
axes[1].imshow(hog_image, cmap='hot')
axes[1].set_title("Biểu đồ hướng cạnh biên HOG (Các đường vạch)")
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Số chiều của vector HOG: {hog_features.shape[0]} chiều.")